# Atlas-Chat-2B — Moroccan CNIE Extraction Test
Gemma-2-2B fine-tuned for Moroccan Darija/Arabic. GGUF ~1.5GB.

**Run Step 1, restart runtime, then run all remaining cells.**

In [ ]:
# Step 1: Install (restart runtime after this)
!pip install llama-cpp-python huggingface-hub
print("\nDONE — Restart runtime, then run next cells.")

In [ ]:
# Step 2: Download + Load
from llama_cpp import Llama
from huggingface_hub import hf_hub_download
import time, json, re, os

VARIANTS = {
    "Atlas-Chat-2B-Q4_K_M": {
        "repo": "QuantFactory/Atlas-Chat-2B-GGUF",
        "file": "Atlas-Chat-2B.Q4_K_M.gguf",
    },
    "Atlas-Chat-2B-Q5_K_M": {
        "repo": "QuantFactory/Atlas-Chat-2B-GGUF",
        "file": "Atlas-Chat-2B.Q5_K_M.gguf",
    },
    "Atlas-Chat-2B-Q6_K": {
        "repo": "QuantFactory/Atlas-Chat-2B-GGUF",
        "file": "Atlas-Chat-2B.Q6_K.gguf",
    },
}

# === CHOOSE VARIANT (uncomment one) ===
ACTIVE = "Atlas-Chat-2B-Q4_K_M"
# ACTIVE = "Atlas-Chat-2B-Q5_K_M"
# ACTIVE = "Atlas-Chat-2B-Q6_K"

v = VARIANTS[ACTIVE]
print(f"Downloading {ACTIVE}...")
t0 = time.time()
model_path = hf_hub_download(repo_id=v["repo"], filename=v["file"])
print(f"Downloaded in {time.time()-t0:.1f}s")

n_threads = os.cpu_count() or 4
print(f"Loading with {n_threads} threads...")
t0 = time.time()
llm = Llama(model_path=model_path, n_ctx=2048, n_threads=n_threads, verbose=False)
print(f"Loaded {ACTIVE} in {time.time()-t0:.1f}s")

In [ ]:
# Step 3: Prompt + Helpers

SYSTEM_PROMPT = """You extract personal data from Moroccan CNIE card OCR output.

IGNORE these template texts: ROYAUME DU MAROC, CARTE NATIONALE D'IDENTITE, المملكة, المغربية, المعربية, البطاقة الوطنية, للتعريف, Né le, مزداد بتاريخ, مزداد بتانيخ, Valable jusqu'au, صالحة الى غاية, المدير العام للأمن الوطني, المدير العام للإمن الوطنى, عبد اللطيف حموشي, عبد اللطيّف حموشي.

NAMES: The card has one last name (family name) and one first name, each in Arabic + French.
- last_name_fr / last_name_ar = family name
- first_name_fr / first_name_ar = given name
- Use your knowledge of Arabic/French Moroccan names to distinguish family from given names.
- French names UPPERCASE.

OTHER FIELDS:
- birth_date: DD.MM.YYYY
- birth_place_fr: city — UPPERCASE
- birth_place_ar: translate birth_place_fr to correct Arabic. NEVER copy Arabic OCR city text.
- expiry_date: DD.MM.YYYY
- card_number: 2 letters + 5-6 digits
- gender: M or F

RULES:
- French text is more reliable than Arabic for names/places
- Fix OCR errors: 1→I, 0→O, 7→T in names/places
- Ignore detections with confidence < 0.5
- Return null (not \"null\") for missing fields
- Return ONLY JSON, no explanation.

Example output:
{"last_name_fr": "ALAOUI", "first_name_fr": "MOHAMMED", "last_name_ar": "العلوي", "first_name_ar": "محمد", "birth_date": "15.06.1990", "birth_place_fr": "CASABLANCA", "birth_place_ar": "الدار البيضاء", "card_number": "CD987654", "expiry_date": "20.06.2030", "gender": "M"}"""


def parse_json(raw):
    cleaned = re.sub(r'<think>.*?</think>', '', raw, flags=re.DOTALL).strip()
    for attempt in [cleaned, re.search(r'\{[^{}]*\}', cleaned, re.DOTALL), re.search(r'\{.*\}', cleaned, re.DOTALL)]:
        try:
            s = attempt.group() if hasattr(attempt, 'group') else attempt
            if s: return json.loads(s)
        except (json.JSONDecodeError, AttributeError):
            continue
    return {"_raw": raw, "_error": "json_parse_failed"}


def extract(ocr_text):
    t0 = time.time()
    messages = [
        {"role": "user", "content": SYSTEM_PROMPT + "\n\nExtract fields:\n\n" + ocr_text},
    ]
    response = llm.create_chat_completion(messages=messages, max_tokens=512, temperature=0.1)
    elapsed = time.time() - t0
    raw = response["choices"][0]["message"]["content"]
    usage = response.get("usage", {})
    print(f"  Time: {elapsed:.2f}s | Tokens: {usage.get('prompt_tokens','?')}→{usage.get('completion_tokens','?')}")
    return {"fields": parse_json(raw), "time_s": round(elapsed, 2), "raw": raw}


def score(extracted, expected):
    correct, details = 0, []
    for field, exp in expected.items():
        got = extracted.get(field)
        e = str(exp).strip().upper() if exp else None
        g = str(got).strip().upper() if got else None
        ok = e == g
        correct += ok
        details.append(f"  {'OK  ' if ok else 'MISS'} {field}: expected '{exp}' got '{got}'")
    return correct, len(expected), details

print("Ready.")

In [ ]:
# Step 4: ALL TESTS

TESTS = {
    "TEST 1 — Clean OCR": {
        "ocr": """
Arabic OCR:
  - text: 'المملكة المغربية', confidence: 0.95
  - text: 'بطاقة التعريف الوطنية', confidence: 0.92
  - text: 'الشافعي', confidence: 0.88
  - text: 'بلال', confidence: 0.91
French OCR:
  - text: 'ROYAUME DU MAROC', confidence: 0.97
  - text: 'CHAFI', confidence: 0.93
  - text: 'BILAL', confidence: 0.95
  - text: 'Ne le 22.01.2007', confidence: 0.89
  - text: 'a RABAT', confidence: 0.92
  - text: 'Valable jusqu au 19.03.2029', confidence: 0.90
  - text: 'AB123456', confidence: 0.94
  - text: 'M', confidence: 0.96""",
        "expected": {"last_name_fr": "CHAFI", "first_name_fr": "BILAL", "last_name_ar": "الشافعي", "first_name_ar": "بلال", "birth_date": "22.01.2007", "birth_place_fr": "RABAT", "birth_place_ar": "الرباط", "card_number": "AB123456", "expiry_date": "19.03.2029", "gender": "M"},
    },
    "TEST 2 — Noisy OCR (0/O, 1/I, 7/T swaps)": {
        "ocr": """
Arabic OCR:
  - text: 'الملكة المكربية', confidence: 0.72
  - text: 'لوح', confidence: 0.55
  - text: 'اجا', confidence: 0.60
French OCR:
  - text: 'R0YAUME DU MAR0C', confidence: 0.75
  - text: 'CHAF1', confidence: 0.70
  - text: 'B1LAL', confidence: 0.72
  - text: 'Ne le 22.O1.20O7', confidence: 0.65
  - text: 'a RABA7', confidence: 0.60
  - text: 'Va1ab1e jusqu au 19.03.2029', confidence: 0.58
  - text: 'A8123456', confidence: 0.55""",
        "expected": {"last_name_fr": "CHAFI", "first_name_fr": "BILAL", "birth_date": "22.01.2007", "birth_place_fr": "RABAT", "birth_place_ar": "الرباط", "card_number": "A8123456", "expiry_date": "19.03.2029"},
    },
    "TEST 3 — Real PaddleOCR output": {
        "ocr": """
OCR detections:
  - text: 'المعربية', confidence: 0.93
  - text: 'المملكة', confidence: 0.93
  - text: 'ROYAUME DU MAROC', confidence: 0.99
  - text: 'للتعريف', confidence: 0.96
  - text: 'البطاقة الوطنية', confidence: 0.97
  - text: 'CARTE NATIONALE D IDENTITE', confidence: 0.99
  - text: 'بلال', confidence: 0.95
  - text: 'BILAL', confidence: 0.99
  - text: 'شافي', confidence: 0.99
  - text: 'CHAFI', confidence: 0.99
  - text: 'Né le', confidence: 0.87
  - text: '22.01.2001', confidence: 0.92
  - text: 'مزداد بتانيخ', confidence: 0.88
  - text: 'الرياة', confidence: 0.71
  - text: 'a RABAT', confidence: 0.90
  - text: 'Valable jusqu au', confidence: 0.99
  - text: '19.03.2029', confidence: 0.99
  - text: 'صالحة الى غاية', confidence: 0.97
  - text: 'AS13538', confidence: 0.99
  - text: 'M', confidence: 0.99
  - text: 'المدير العام للإمن الوطنى', confidence: 0.97
  - text: 'Thy', confidence: 0.53
  - text: 'عبد اللطيّف حموشي', confidence: 0.91""",
        "expected": {"last_name_fr": "CHAFI", "first_name_fr": "BILAL", "last_name_ar": "شافي", "first_name_ar": "بلال", "birth_date": "22.01.2001", "birth_place_fr": "RABAT", "birth_place_ar": "الرباط", "card_number": "AS13538", "expiry_date": "19.03.2029", "gender": "M"},
    },
    "TEST 4 — Compound name + Casablanca": {
        "ocr": """
OCR detections:
  - text: 'المملكة', confidence: 0.91
  - text: 'المعربية', confidence: 0.89
  - text: 'ROYAUME DU MAROC', confidence: 0.99
  - text: 'البطاقة الوطنية', confidence: 0.95
  - text: 'CARTE NATIONALE D IDENTITE', confidence: 0.99
  - text: 'عبد الرحمن', confidence: 0.87
  - text: 'ABDERRAHMANE', confidence: 0.98
  - text: 'البكاوي', confidence: 0.82
  - text: 'EL BAKAOUI', confidence: 0.97
  - text: 'Né le', confidence: 0.90
  - text: '15.06.1985', confidence: 0.99
  - text: 'الداز البيظاء', confidence: 0.62
  - text: 'a CASABLANCA', confidence: 0.96
  - text: 'Valable jusqu au', confidence: 0.98
  - text: '01.07.2030', confidence: 0.99
  - text: 'BK987654', confidence: 0.99
  - text: 'M', confidence: 0.98
  - text: 'المدير العام للإمن الوطنى', confidence: 0.96
  - text: 'عبد اللطيّف حموشي', confidence: 0.90""",
        "expected": {"last_name_fr": "EL BAKAOUI", "first_name_fr": "ABDERRAHMANE", "last_name_ar": "البكاوي", "first_name_ar": "عبد الرحمن", "birth_date": "15.06.1985", "birth_place_fr": "CASABLANCA", "birth_place_ar": "الدار البيضاء", "card_number": "BK987654", "expiry_date": "01.07.2030", "gender": "M"},
    },
    "TEST 5 — Female + Oujda + low-confidence noise": {
        "ocr": """
OCR detections:
  - text: 'المملكة', confidence: 0.90
  - text: 'المعربية', confidence: 0.88
  - text: 'ROYAUME DU MAROC', confidence: 0.99
  - text: 'للتعريف', confidence: 0.93
  - text: 'البطاقة الوطنية', confidence: 0.96
  - text: 'CARTE NATIONALE D IDENTITE', confidence: 0.99
  - text: 'فاطمة', confidence: 0.91
  - text: 'FATIMA', confidence: 0.99
  - text: 'الزهراء', confidence: 0.78
  - text: 'EZZAHRA', confidence: 0.45
  - text: 'بنيسى', confidence: 0.74
  - text: 'BENNISSI', confidence: 0.98
  - text: 'Né le', confidence: 0.88
  - text: '03.11.1992', confidence: 0.99
  - text: 'وخدة', confidence: 0.55
  - text: 'a OUJDA', confidence: 0.94
  - text: 'Valable jusqu au', confidence: 0.99
  - text: '10.12.2028', confidence: 0.99
  - text: 'CD554433', confidence: 0.99
  - text: 'F', confidence: 0.99
  - text: 'Xr4', confidence: 0.35
  - text: 'المدير العام للإمن الوطنى', confidence: 0.95
  - text: 'عبد اللطيّف حموشي', confidence: 0.89""",
        "expected": {"last_name_fr": "BENNISSI", "first_name_fr": "FATIMA", "last_name_ar": "بنيسى", "first_name_ar": "فاطمة", "birth_date": "03.11.1992", "birth_place_fr": "OUJDA", "birth_place_ar": "وجدة", "card_number": "CD554433", "expiry_date": "10.12.2028", "gender": "F"},
    },
    "TEST 6 — Minimal detections, missing fields": {
        "ocr": """
OCR detections:
  - text: 'ROYAUME DU MAROC', confidence: 0.99
  - text: 'CARTE NATIONALE D IDENTITE', confidence: 0.99
  - text: 'محمد', confidence: 0.93
  - text: 'MOHAMMED', confidence: 0.99
  - text: 'ALAOUI', confidence: 0.98
  - text: '22.08.1970', confidence: 0.97
  - text: 'a FES', confidence: 0.92
  - text: 'EE112233', confidence: 0.99
  - text: 'M', confidence: 0.99""",
        "expected": {"last_name_fr": "ALAOUI", "first_name_fr": "MOHAMMED", "last_name_ar": None, "first_name_ar": "محمد", "birth_date": "22.08.1970", "birth_place_fr": "FES", "birth_place_ar": "فاس", "card_number": "EE112233", "expiry_date": None, "gender": "M"},
    },
}

# Run all tests
summary = []
for name, t in TESTS.items():
    print("=" * 60)
    print(f"{name} — {ACTIVE}")
    print("=" * 60)
    result = extract(t["ocr"])
    print(json.dumps(result["fields"], indent=2, ensure_ascii=False))
    c, total, details = score(result["fields"], t["expected"])
    for d in details:
        print(d)
    print(f"Score: {c}/{total} | Time: {result['time_s']}s\n")
    summary.append((name, c, total, result["time_s"]))

print("\n" + "=" * 60)
print(f"SUMMARY — {ACTIVE}")
print("=" * 60)
total_correct = sum(s[1] for s in summary)
total_fields = sum(s[2] for s in summary)
total_time = sum(s[3] for s in summary)
for name, c, t, ts in summary:
    print(f"  {c:2d}/{t:2d} | {ts:5.1f}s | {name}")
print(f"\nOVERALL: {total_correct}/{total_fields} fields correct | {total_time:.1f}s total")